# Метрики регресії · практика> Лекція: [lecture.html](lecture.html) · Домашнє завдання: [homework.html](homework.html) · Тест: [quiz.html](quiz.html)Наскрізний приклад той самий, що й у лекції — **передбачення ціни вживаного телефоназа оголошенням**. Колонки: модель, рік, стан, обсяг памʼяті, ціна.Що зробимо:1. порахуємо MAE, MSE, RMSE, R², MAPE і SMAPE **вручну на NumPy** — рядок за рядком;2. звіримо кожне число зі `sklearn.metrics` — вони мають зійтися до останнього знака;3. зіпсуємо дані одним викидом і подивимось, яка метрика як зреагувала;4. побудуємо дві моделі, на яких MAE і RMSE дають **протилежні** вердикти;5. проженемо все на більшій вибірці зі справжньою лінійною регресією.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             r2_score, mean_absolute_percentage_error)

# фіксуємо генератор випадкових чисел, щоб результат був однаковий у всіх
генератор = np.random.default_rng(42)

print("numpy", np.__version__)
print("pandas", pd.__version__)
print("готово — усі бібліотеки на місці")

## 1 · Шість оголошень із лекціїПочинаємо з крихітної таблиці, яку можна перевірити калькулятором. Це важливо:поки числа маленькі, кожну формулу видно наскрізь.Колонка **ціна** — це факт (за скільки телефон реально продали), колонка**прогноз** — те, що назвала модель.

In [ ]:
оголошення = pd.DataFrame({
    "модель":   ["Redmi Note 10", "Samsung A52", "Poco X3",
                 "iPhone SE 2020", "Realme 8", "iPhone 11"],
    "рік":      [2021, 2021, 2020, 2020, 2021, 2019],
    "стан":     ["добрий", "добрий", "задовільний", "добрий", "відмінний", "відмінний"],
    "памʼять, ГБ": [128, 128, 64, 128, 128, 64],
    "ціна":     [5200, 7300, 3900, 8100, 6100, 12000],
    "прогноз":  [5600, 6700, 4800, 8500, 6200, 10800],
})

print(оголошення.to_string(index=False))

## 2 · Помилка одного рядка й чому суму брати не можнаПомилка (residual) — це `факт − прогноз`. Знак має сенс: мінус означає, що модельпереоцінила телефон, плюс — що недооцінила.Зараз побачимо головну пастку: сума помилок дорівнює нулю, хоча модель промахнуласьна **кожному** рядку.

In [ ]:
факт = оголошення["ціна"].to_numpy(dtype=float)
прогноз = оголошення["прогноз"].to_numpy(dtype=float)

# помилка окремо для кожного оголошення — щоб побачити не одне число, а розподіл
помилка = факт - прогноз
оголошення["помилка"] = помилка.astype(int)

print(оголошення[["модель", "ціна", "прогноз", "помилка"]].to_string(index=False))
print()
print("сума помилок      :", помилка.sum())
print("сума модулів      :", np.abs(помилка).sum())
print()
print("Сума нульова — але жоден прогноз не влучив. Плюси й мінуси знищили одне одного.")

## 3 · MAE — середня абсолютна помилкаФормула: `MAE = (1/n) · Σ |факт − прогноз|`.Прибираємо знак модулем, складаємо, ділимо на кількість оголошень. Найпростішаметрика з усіх — і найлегша для пояснення замовникові.

In [ ]:
# рахуємо руками, крок за кроком — саме так, як у лекції
модулі_помилок = np.abs(помилка)
сума_модулів = модулі_помилок.sum()
наш_mae = сума_модулів / len(факт)

бібліотечний_mae = mean_absolute_error(факт, прогноз)

print("модулі помилок    :", модулі_помилок.astype(int))
print("сума модулів      :", сума_модулів)
print("поділити на", len(факт), "        :", наш_mae)
print()
print("наш MAE           :", наш_mae)
print("sklearn MAE       :", бібліотечний_mae)

assert np.allclose(наш_mae, бібліотечний_mae), "розрахунок розійшовся!"
print("✅ збігається — модель помиляється в середньому на 600 грн")

## 4 · MSE і RMSE — те саме, але через квадрат`MSE = (1/n) · Σ (факт − прогноз)²` — середній квадрат помилки. Число виходить у«гривнях у квадраті», тому його не можна прочитати вголос.`RMSE = √MSE` повертає нас у гривні. Зверни увагу: RMSE вийде **більшою** за MAE,бо квадрат непропорційно роздуває великі промахи.

In [ ]:
квадрати_помилок = помилка ** 2
наш_mse = квадрати_помилок.mean()
наш_rmse = np.sqrt(наш_mse)

бібліотечний_mse = mean_squared_error(факт, прогноз)

print("квадрати помилок  :", квадрати_помилок.astype(int))
print("сума квадратів    :", квадрати_помилок.sum())
print()
print("наш MSE           :", наш_mse, "(гривень у квадраті — не читається)")
print("sklearn MSE       :", бібліотечний_mse)
print("наш RMSE          :", наш_rmse, "грн")
print()
# найбільший квадрат показує, скільки важить один-єдиний поганий прогноз
частка_найбільшого = квадрати_помилок.max() / квадрати_помилок.sum()
print(f"найгірший рядок дає {частка_найбільшого:.1%} усієї суми квадратів")

assert np.allclose(наш_mse, бібліотечний_mse), "розрахунок розійшовся!"
print("✅ збігається")

## 5 · R² — порівняння з «завжди називаю середнє»MAE в гривнях не каже, добре це чи погано: усе залежить від масштабу цін. R² прибираєодиниці, порівнюючи нашу модель із найдурнішою можливою — тією, що завжди називаєсередню ціну.`R² = 1 − SSres / SStot`, де `SSres = Σ(факт − прогноз)²`, а `SStot = Σ(факт − середнє)²`.

In [ ]:
середня_ціна = факт.mean()

ss_res = ((факт - прогноз) ** 2).sum()          # помилка нашої моделі
ss_tot = ((факт - середня_ціна) ** 2).sum()      # помилка моделі «завжди середнє»
наш_r2 = 1 - ss_res / ss_tot

бібліотечний_r2 = r2_score(факт, прогноз)

print("середня ціна      :", середня_ціна)
print("SSres (наша модель):", ss_res)
print("SStot (середнє)   :", ss_tot)
print()
print("наш R²            :", наш_r2)
print("sklearn R²        :", бібліотечний_r2)

assert np.allclose(наш_r2, бібліотечний_r2), "розрахунок розійшовся!"
print(f"✅ збігається — модель пояснює {наш_r2:.1%} розкиду цін")

### R² буває відʼємнимЦе не помилка й не рідкість. Якщо модель гірша за просте середнє, `SSres > SStot`, ідріб перевищує одиницю. Перевіримо на моделі, яка вперто називає 10 000 грн для всіх.

In [ ]:
уперта_модель = np.full(len(факт), 10000.0)   # завжди 10 000 грн, хай там що

print("R² моделі «завжди середнє»:", round(r2_score(факт, np.full(len(факт), середня_ціна)), 4))
print("R² моделі «завжди 10 000» :", round(r2_score(факт, уперта_модель), 4))
print()
print("Нуль означає «не краще за середнє», відʼємне число — «гірше за середнє».")

## 6 · MAPE і SMAPE — відсотки замість гривень`MAPE = (1/n) · Σ |факт − прогноз| / факт` — помилку кожного рядка ділимо на **йоговласну** ціну. Так промах у 600 грн на телефоні за 12 000 і на телефоні за 1 500перестають бути однаковими.SMAPE ділить не на факт, а на середнє між фактом і прогнозом. У `sklearn` її немає —напишемо самі.

In [ ]:
частки = np.abs(помилка) / факт          # відносна помилка кожного рядка
наш_mape = частки.mean()

бібліотечний_mape = mean_absolute_percentage_error(факт, прогноз)

# SMAPE: у знаменнику середнє між фактом і прогнозом, тому нуль у факті її не ламає
наш_smape = (np.abs(помилка) / ((факт + прогноз) / 2)).mean()

таблиця_відсотків = оголошення[["модель", "ціна", "помилка"]].copy()
таблиця_відсотків["частка"] = (частки * 100).round(2)
print(таблиця_відсотків.to_string(index=False))
print()
print(f"наш MAPE          : {наш_mape:.4%}")
print(f"sklearn MAPE      : {бібліотечний_mape:.4%}")
print(f"наш SMAPE         : {наш_smape:.4%}")

assert np.allclose(наш_mape, бібліотечний_mape), "розрахунок розійшовся!"
print("✅ збігається")
print()
print("Найгірший рядок за MAPE — Poco X3 (23%), хоча за модулем помилки")
print("найгірший був iPhone 11. Метрики питають різне.")

## 7 · Одна функція, яка рахує всеДалі метрики знадобляться нам багато разів, тому зберемо їх в одну функцію.Вона повертає звичайний словник — його зручно класти в `DataFrame`.

In [ ]:
def усі_метрики(факт, прогноз):
    # Рахує пʼять метрик регресії й повертає їх словником.
    # MAPE ставимо None, якщо серед фактів є нуль: ділення на нуль означає,
    # що метрика просто не існує, а не що вона дорівнює нулю.
    помилка = факт - прогноз
    результат = {
        "MAE": np.abs(помилка).mean(),
        "MSE": (помилка ** 2).mean(),
        "RMSE": np.sqrt((помилка ** 2).mean()),
        "R2": r2_score(факт, прогноз),
        "SMAPE, %": (np.abs(помилка) / ((np.abs(факт) + np.abs(прогноз)) / 2)).mean() * 100,
    }
    if np.any(факт == 0):
        результат["MAPE, %"] = None
    else:
        результат["MAPE, %"] = (np.abs(помилка) / факт).mean() * 100
    return результат


базові_метрики = усі_метрики(факт, прогноз)
for назва, значення in базові_метрики.items():
    print(f"{назва:<10} {значення:>12.4f}")

## 8 · Одне шахрайське оголошення за 1 грнТепер найцікавіше. Додамо сьомий рядок: телефон, виставлений за **1 грн** — типовешахрайське оголошення «пишіть у месенджер». Модель, дивлячись на характеристики,оцінила його у 6 000 грн.Модель ми не переучуємо. Змінюються лише дані — і подивимось, що станеться з кожною метрикою.

In [ ]:
факт_із_викидом = np.append(факт, 1.0)          # ціна оголошення — 1 гривня
прогноз_із_викидом = np.append(прогноз, 6000.0)  # модель нічого не запідозрила

метрики_з_викидом = усі_метрики(факт_із_викидом, прогноз_із_викидом)

порівняння = pd.DataFrame({
    "шість чесних": базові_метрики,
    "плюс 1 грн": метрики_з_викидом,
})
порівняння["у скільки разів"] = порівняння["плюс 1 грн"] / порівняння["шість чесних"]

print(порівняння.round(3).to_string())
print()
print("Один рядок із семи. MAE зросла у 2,3 раза, RMSE — у 3,4 раза,")
print("а MAPE злетіла в тисячі разів — бо ділиться на ціну в 1 гривню.")

In [ ]:
# а що буде, якщо ціна взагалі нульова? MAPE перестає існувати
факт_із_нулем = np.append(факт, 0.0)
прогноз_із_нулем = np.append(прогноз, 6000.0)

метрики_з_нулем = усі_метрики(факт_із_нулем, прогноз_із_нулем)
print("MAPE при нульовій ціні :", метрики_з_нулем["MAPE, %"], "← ділення на нуль")
print("RMSE при нульовій ціні :", round(метрики_з_нулем["RMSE"], 1), "грн — рахується спокійно")

## 9 · Дві моделі, два протилежні вердиктиНайважливіший експеримент теми. Візьмемо ті самі шість оголошень і дві різні моделі:* **Обережна** помиляється на кожному оголошенні рівно на 650 грн;* **Ризикова** влучає майже точно на пʼяти (промах 200 грн), але на iPhone 11  промахується на 2 000 грн.MAE і RMSE порахують чесно — і назвуть різних переможців.

In [ ]:
обережна = np.array([4550, 7950, 3250, 8750, 5450, 12650], dtype=float)
ризикова = np.array([5000, 7500, 3700, 8300, 5900, 14000], dtype=float)

дві_моделі = pd.DataFrame({
    "Обережна": усі_метрики(факт, обережна),
    "Ризикова": усі_метрики(факт, ризикова),
})
print(дві_моделі.round(3).to_string())
print()

краща_за_mae = дві_моделі.loc["MAE"].idxmin()
краща_за_rmse = дві_моделі.loc["RMSE"].idxmin()
print(f"MAE  обирає : {краща_за_mae}")
print(f"RMSE обирає : {краща_за_rmse}")
print()
if краща_за_mae != краща_за_rmse:
    print("Вердикти протилежні. Метрика не знаходить «кращу модель» —")
    print("вона виконує те визначення кращого, яке ти в неї заклав.")

## 10 · Велика вибірка й справжня модельШість рядків добре для розуміння, але метрики міряють на сотнях. Згенеруємо 300оголошень із тими самими колонками, навчимо лінійну регресію й порахуємо метрикина тестовій частині — тобто на даних, яких модель не бачила.

In [ ]:
кількість = 300

назви_моделей = генератор.choice(
    ["Redmi Note 10", "Samsung A52", "Poco X3", "iPhone SE 2020", "Realme 8", "iPhone 11"],
    size=кількість)
роки = генератор.integers(2016, 2024, size=кількість)
стани = генератор.choice(["задовільний", "добрий", "відмінний"],
                         size=кількість, p=[0.25, 0.5, 0.25])
обсяги = генератор.choice([64, 128, 256], size=кількість)

# стан переводимо в число: моделі потрібні числа, а не слова
бал_стану = pd.Series(стани).map({"задовільний": 0, "добрий": 1, "відмінний": 2}).to_numpy()

# «справжня» залежність, яку модель має відшукати, плюс шум ринку
ціни = (900
        + 1400 * (роки - 2016)
        + 11 * обсяги
        + 700 * бал_стану
        + генератор.normal(0, 900, size=кількість))
ціни = np.clip(ціни, 500, None).round(-2)     # ціни в оголошеннях круглі

база = pd.DataFrame({
    "модель": назви_моделей, "рік": роки, "стан": стани,
    "памʼять, ГБ": обсяги, "ціна": ціни,
})
print(база.head(8).to_string(index=False))
print()
print("рядків:", len(база), "· ціна від", int(ціни.min()), "до", int(ціни.max()), "грн")

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

ознаки = np.column_stack([роки - 2016, обсяги, бал_стану])

ознаки_навч, ознаки_тест, ціни_навч, ціни_тест = train_test_split(
    ознаки, ціни, test_size=0.3, random_state=42)

модель = LinearRegression().fit(ознаки_навч, ціни_навч)
прогноз_тест = модель.predict(ознаки_тест)

print("коефіцієнти моделі:")
for назва, вага in zip(["рік (від 2016)", "памʼять, ГБ", "бал стану"], модель.coef_):
    print(f"  {назва:<16} {вага:8.1f} грн")
print(f"  вільний член     {модель.intercept_:8.1f} грн")
print()
print("рядків у навчальній вибірці:", len(ціни_навч), "· у тестовій:", len(ціни_тест))

In [ ]:
метрики_тесту = усі_метрики(ціни_тест, прогноз_тест)
for назва, значення in метрики_тесту.items():
    print(f"{назва:<10} {значення:>12.3f}")
print()
print(f"Читається так: модель помиляється в середньому на {метрики_тесту['MAE']:.0f} грн")
print(f"і пояснює {метрики_тесту['R2']:.1%} розкиду цін на даних, яких не бачила.")

### Розподіл помилок каже більше, ніж одне числоГістограма залишків — найдешевша діагностика, яка є. Здорова модель дає симетричнухмару навколо нуля. Хвіст в один бік означає, що модель систематично завищуєабо занижує ціну.

In [ ]:
залишки = ціни_тест - прогноз_тест

фігура, осі = plt.subplots(1, 2, figsize=(11, 4))

осі[0].hist(залишки, bins=20, color="#c2185b", alpha=0.75)
осі[0].axvline(0, color="#17212b", linewidth=1.2)
осі[0].set_title("Розподіл помилок на тесті")
осі[0].set_xlabel("факт − прогноз, грн")
осі[0].set_ylabel("кількість оголошень")

осі[1].scatter(прогноз_тест, ціни_тест, s=18, color="#0f766e", alpha=0.7)
межі = [ціни_тест.min(), ціни_тест.max()]
осі[1].plot(межі, межі, color="#17212b", linewidth=1.2)
осі[1].set_title("Факт проти прогнозу")
осі[1].set_xlabel("прогноз, грн")
осі[1].set_ylabel("факт, грн")

plt.tight_layout()
plt.show()

print(f"середня помилка (перекіс): {залишки.mean():.1f} грн")
print(f"медіанний модуль помилки : {np.median(np.abs(залишки)):.1f} грн")
print(f"MAE                      : {np.abs(залишки).mean():.1f} грн")
print()
print("Медіана менша за MAE — отже кілька великих промахів тягнуть середнє вгору.")

### Що зробить із цими метриками один викидПовторимо експеримент із лекції на великій вибірці: зіпсуємо **одне** оголошенняз тестової частини, виставивши ціну 1 грн. Модель не переучуємо.

In [ ]:
ціни_зіпсовані = ціни_тест.copy()
ціни_зіпсовані[0] = 1.0                # одне шахрайське оголошення з 90

до = усі_метрики(ціни_тест, прогноз_тест)
після = усі_метрики(ціни_зіпсовані, прогноз_тест)

ефект = pd.DataFrame({"до": до, "після одного викиду": після})
ефект["у скільки разів"] = ефект["після одного викиду"] / ефект["до"]
print(ефект.round(3).to_string())
print()
print("Один рядок із", len(ціни_тест), "— а MAPE вже непридатна для звіту.")

## Завдання### 🟢 Рівень 1 — БазаДодай до функції `усі_метрики` шосту метрику — **медіанну абсолютну помилку**(`np.median(np.abs(факт - прогноз))`). Прожени її через таблицюз розділу 8 (шість чесних оголошень проти семи з викидом за 1 грн) і скажи словами,чому вона майже не зрушила.**Зроблено, якщо:** у таблиці «шість чесних / плюс 1 грн» медіанна абсолютна помилказмінилась менш ніж у 1,5 раза, тоді як RMSE — щонайменше вдвічі.### 🟡 Рівень 2 — ПлюсПобудуй графік: як змінюються MAE, RMSE і MAPE, коли ціну одного тестовогооголошення поступово піднімати від 0 до 30 000 грн. Модель не переучуй.Підказка: цикл по значеннях ціни, у кожному кроці — виклик `усі_метрики`.**Зроблено, якщо:** на графіку видно, що MAE росте прямою лінією, RMSE загинаєтьсявгору, а MAPE має різкий пік біля нуля й майже пласка праворуч.### 🔴 Рівень 3 — ВикликНавчи дві моделі на одних і тих самих даних так, щоб вони дали **протилежні**вердикти за MAE й RMSE. Найпростіший шлях: звичайна `LinearRegression`проти `sklearn.linear_model.HuberRegressor` на даних, куди ти додав кілька викидів.**Зроблено, якщо:** одна модель має меншу MAE, друга — меншу RMSE, і ти письмовопояснив, яку з них узяв би для сервісу оцінки телефонів і чому.